# Algebraic Characteristic Polynomial for MELG

## Formula

The MELG state is $(w_0, w_1, \ldots, w_{s-1}, v)$ where $s = N-1$ and each word is $w$ bits.
One canonical step (pointer at 0, then rotate) gives:

$$x = U \cdot w_0 + L \cdot w_1, \quad
v_{\text{new}} = A \cdot x + w_M + B_1 \cdot v, \quad
w_0' = x + B_2 \cdot v_{\text{new}}$$

The eigenvalue equation ($w_j = t^j w_0$, $v_{\text{new}} = t \cdot v$) gives a $2w \times 2w$ system:

$$R(t) \begin{pmatrix} w_0 \\ v \end{pmatrix} = 0$$

$$R(t) = \begin{pmatrix} R_{11} & R_{12} \\ R_{21} & R_{22} \end{pmatrix}$$

where:
- $R_{11} = t^s I + (I + B_2 A)(U + tL) + t^M B_2$
- $R_{12} = B_2 B_1$
- $R_{21} = A(U + tL) + t^M I$
- $R_{22} = (t+1)I + S_L(\sigma_1)$

The characteristic polynomial is:
$$\phi(t) = \det(R(t)) \,/\, t^r$$

## Schur Complement Optimization

Since $R_{22}$ has low-degree entries and $S_L$ is nilpotent:
$$\det(R_{22}) = (t+1)^w, \quad R_{22}^{-1} = \sum_{j=0}^{q} (t+1)^{-(j+1)} S_L^j$$
where $q = \lfloor(w-1)/\sigma_1\rfloor$.

Clearing denominators with $Q = \sum_{j=0}^{q} (t+1)^{q-j} S_L^j$:
$$S' = (t+1)^{q+1} R_{11} - R_{12} \cdot Q \cdot R_{21}$$
$$\phi(t) = \det(S') \,/\, ((t+1)^{wq} \cdot t^r)$$

This reduces the Bareiss determinant from $2w \times 2w$ to $w \times w$.

In [ ]:
import time
from regpoly.core.generator import Generator

TABLE_I = [
    {"name": "MELG607-64", "p": 607, "w": 64, "r": 33, "N": 10,
     "M": 5, "sigma1": 13, "sigma2": 35, "a": 0x81f1fd68012348bc, "N1": 313},
    {"name": "MELG1279-64", "p": 1279, "w": 64, "r": 1, "N": 20,
     "M": 7, "sigma1": 22, "sigma2": 37, "a": 0x1afefd1526d3952b, "N1": 641},
    {"name": "MELG2281-64", "p": 2281, "w": 64, "r": 23, "N": 36,
     "M": 17, "sigma1": 36, "sigma2": 21, "a": 0x7cbe23ebca8a6d36, "N1": 1145},
    {"name": "MELG4253-64", "p": 4253, "w": 64, "r": 35, "N": 67,
     "M": 29, "sigma1": 30, "sigma2": 20, "a": 0xfac1e8c56471d722, "N1": 2129},
    {"name": "MELG11213-64", "p": 11213, "w": 64, "r": 51, "N": 176,
     "M": 45, "sigma1": 33, "sigma2": 13, "a": 0xddbcd6e525e1c757, "N1": 5455},
    {"name": "MELG19937-64", "p": 19937, "w": 64, "r": 31, "N": 312,
     "M": 81, "sigma1": 23, "sigma2": 33, "a": 0x5c32e06df730fc42, "N1": 9603},
    {"name": "MELG44497-64", "p": 44497, "w": 64, "r": 47, "N": 696,
     "M": 373, "sigma1": 37, "sigma2": 14, "a": 0x4fa9ca36f293c9a9, "N1": 19475},
]

print(f"{'Name':<16} {'k':>6} {'N1':>6} {'Expected':>8} {'Time':>8}")
print("-" * 50)

for g in TABLE_I:
    gen = Generator.create("MELG", g["w"],
        w=g["w"], N=g["N"], M=g["M"], r=g["r"],
        sigma1=g["sigma1"], sigma2=g["sigma2"], a=g["a"])
    
    t0 = time.time()
    cp = gen.char_poly()
    elapsed = time.time() - t0
    
    hw = bin(cp._val).count('1') + 1
    status = "OK" if hw == g["N1"] else f"FAIL (got {hw})"
    print(f"{g['name']:<16} {gen.k:>6} {hw:>6} {g['N1']:>8} {elapsed:>7.1f}s  {status}")